# Firefox Launcher Debug

This notebook will help us debug the frontend-backend interaction for the Firefox launcher extension.

In [5]:
## 1. Test Server Proxy Entry Point

import sys
sys.path.append('/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher')

from jupyterlab_firefox_launcher.server_proxy import launch_firefox

# Test the entry point function
print("Testing launch_firefox() function...")
config = launch_firefox()
print(f"Config keys: {list(config.keys())}")
print(f"Timeout: {config['timeout']}")
print(f"Port: {config['port']}")
print(f"Map path: {config['mappath']}")

# Test command generation
if callable(config.get('command')):
    cmd = config['command']()
    print(f"\nCommand length: {len(cmd)} arguments")
    print(f"First 5 args: {cmd[:5]}")
    print(f"Last 5 args: {cmd[-5:]}")
else:
    print("ERROR: Command is not callable!")

Testing launch_firefox() function...
Config keys: ['command', 'timeout', 'new_browser_tab', 'launcher_entry', 'port', 'mappath', 'request_headers_override', 'ready_check']
Timeout: 60
Port: 0
Map path: {'/': '/index.html'}

Command length: 48 arguments
First 5 args: ['xpra', 'start', '--bind-tcp=0.0.0.0:{port}', '--html=on', '--daemon=no']
Last 5 args: ['--dbus-launch=', '--dbus-proxy=no', '--remote-logging=no', '--bandwidth-detection=no', '--pings=yes']


In [6]:
## 2. Test Proxy Endpoint Availability

import requests
import json

# Test the proxy endpoint directly
proxy_url = "http://localhost:8888/proxy/firefox-desktop"
print(f"Testing: {proxy_url}")

try:
    # First, test with HEAD request
    response = requests.head(proxy_url, timeout=5)
    print(f"HEAD request: {response.status_code}")
    print(f"Headers: {dict(response.headers)}")
    
    if response.status_code == 404:
        print("❌ 404 - Proxy endpoint not found")
        print("This suggests jupyter-server-proxy is not recognizing our entry point")
    elif response.status_code == 502:
        print("⚠️  502 - Proxy found but backend failed to start")
        print("This suggests Xpra command is failing")
    else:
        print(f"✅ Response: {response.status_code}")
        
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

# Also test the base proxy path
try:
    response = requests.get("http://localhost:8888/proxy/", timeout=2)
    print(f"\nBase proxy path status: {response.status_code}")
    if response.status_code == 200:
        print("Base proxy is working")
except requests.exceptions.RequestException as e:
    print(f"Base proxy failed: {e}")

Testing: http://localhost:8888/proxy/firefox-desktop
HEAD request: 404
Headers: {'Server': 'TornadoServer/6.5.1', 'Content-Type': 'text/html', 'Date': 'Tue, 22 Jul 2025 00:26:34 GMT', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "frame-ancestors 'self'; report-uri /api/security/csp-report", 'Content-Length': '2957', 'Set-Cookie': 'username-localhost-8888="2|1:0|10:1753143994|23:username-localhost-8888|200:eyJ1c2VybmFtZSI6ICI5YWRiNDViNmJkOTA0ZDQ5YmI3M2YxMGYyM2M3YmQ0YSIsICJuYW1lIjogIkFub255bW91cyBQYXNpcGhhZSIsICJkaXNwbGF5X25hbWUiOiAiQW5vbnltb3VzIFBhc2lwaGFlIiwgImluaXRpYWxzIjogIkFQIiwgImNvbG9yIjogbnVsbH0=|583bb5c997b72e54286671164e6fac813e4d09be642c88a51e620b9199af76c6"; expires=Thu, 21 Aug 2025 00:26:34 GMT; HttpOnly; Path=/, _xsrf=2|11a52e44|91f0ac11841682c6b3128926a9ec3459|1753143994; expires=Thu, 21 Aug 2025 00:26:34 GMT; Path=/'}
❌ 404 - Proxy endpoint not found
This suggests jupyter-server-proxy is not recognizing our entry point

Base proxy path status: 404


In [7]:
## 3. Test Minimal Xpra Command

import subprocess
import time
import threading
import socket

def test_minimal_xpra():
    """Test if a minimal xpra command works"""
    
    # Test basic xpra availability
    try:
        result = subprocess.run(['xpra', '--version'], capture_output=True, text=True, timeout=5)
        print(f"Xpra version: {result.stdout.strip()}")
    except Exception as e:
        print(f"❌ Xpra not available: {e}")
        return False
    
    # Try a minimal command that should work
    port = 9997
    cmd = [
        'xpra', 'start', 
        f'--bind-tcp=0.0.0.0:{port}',
        '--html=on',
        '--daemon=no',
        '--exit-with-children=yes',
        '--start-child=xterm'  # Use xterm instead of our complex firefox wrapper
    ]
    
    print(f"Testing minimal command on port {port}:")
    print(" ".join(cmd))
    
    try:
        # Start the process in background
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        
        # Wait a bit for startup
        time.sleep(3)
        
        # Check if port is listening
        try:
            sock = socket.create_connection(('127.0.0.1', port), timeout=1)
            sock.close()
            print(f"✅ Port {port} is listening")
            result = True
        except ConnectionRefusedError:
            print(f"❌ Port {port} is not listening")
            result = False
            
        # Kill the process
        proc.terminate()
        proc.wait(timeout=5)
        
        # Show any output
        stdout, stderr = proc.communicate()
        if stdout:
            print(f"STDOUT: {stdout[:200]}...")
        if stderr:
            print(f"STDERR: {stderr[:200]}...")
            
        return result
        
    except Exception as e:
        print(f"❌ Failed to run minimal xpra: {e}")
        return False

# Run the test
print("Testing minimal Xpra functionality...")
test_minimal_xpra()

Testing minimal Xpra functionality...
Xpra version: xpra v3.1.5
Testing minimal command on port 9997:
xpra start --bind-tcp=0.0.0.0:9997 --html=on --daemon=no --exit-with-children=yes --start-child=xterm
✅ Port 9997 is listening
STDERR: 2025-07-22 00:26:43,352 created tcp socket '0.0.0.0:9997'
2025-07-22 00:26:43,353 cannot access python uinput module:
2025-07-22 00:26:43,353  No module named 'uinput'
_XSERVTransSocketUNIXCreateListe...


True

# Firefox Launcher Extension Debug

This notebook tests and verifies the frontend-backend interaction for the Xpra Firefox launcher extension.

In [8]:
# Import Required Libraries
import sys
import os
import requests
import socket
import time
from pathlib import Path
import pkg_resources
from urllib.parse import urljoin

# Add the project directory to path so we can import our modules
sys.path.insert(0, '/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher')

try:
    from jupyterlab_firefox_launcher.server_proxy import launch_firefox, create_xpra_command, get_firefox_command, check_xpra_ready
    print("✅ Successfully imported server_proxy functions")
except ImportError as e:
    print(f"❌ Failed to import server_proxy functions: {e}")
    print("Let's check what's available...")
    import jupyterlab_firefox_launcher
    print(f"Available in module: {dir(jupyterlab_firefox_launcher)}")

print(f"Current working directory: {os.getcwd()}")
print(f"Python path: {sys.path[:3]}...")  # Show first 3 entries

✅ Successfully imported server_proxy functions
Current working directory: /home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher
Python path: ['/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher', '/usr/lib/python312.zip', '/usr/lib/python3.12']...


/tmp/ipykernel_307385/355376662.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [9]:
# Test Jupyter Server Proxy Configuration
print("=== Testing Jupyter Server Proxy Configuration ===")

try:
    config = launch_firefox()
    print("✅ launch_firefox() function works")
    print(f"Configuration keys: {list(config.keys())}")
    
    # Test each configuration parameter
    print("\n📋 Configuration Details:")
    print(f"  • timeout: {config.get('timeout')}")
    print(f"  • new_browser_tab: {config.get('new_browser_tab')}")
    print(f"  • port: {config.get('port')}")
    print(f"  • launcher_entry enabled: {config.get('launcher_entry', {}).get('enabled')}")
    print(f"  • mappath: {config.get('mappath')}")
    print(f"  • request_headers_override: {config.get('request_headers_override')}")
    print(f"  • command callable: {callable(config.get('command'))}")
    print(f"  • ready_check callable: {callable(config.get('ready_check'))}")
    
except Exception as e:
    print(f"❌ Error testing launch_firefox(): {e}")
    import traceback
    traceback.print_exc()

=== Testing Jupyter Server Proxy Configuration ===
✅ launch_firefox() function works
Configuration keys: ['command', 'timeout', 'new_browser_tab', 'launcher_entry', 'port', 'mappath', 'request_headers_override', 'ready_check']

📋 Configuration Details:
  • timeout: 60
  • new_browser_tab: False
  • port: 0
  • launcher_entry enabled: False
  • mappath: {'/': '/index.html'}
  • request_headers_override: {'X-Forwarded-Proto': 'http'}
  • command callable: True
  • ready_check callable: True


In [10]:
# Test Firefox Command Creation
print("=== Testing Firefox Command Creation ===")

# Test get_firefox_command()
try:
    xpra, firefox = get_firefox_command()
    print(f"✅ Found dependencies:")
    print(f"  • Xpra: {xpra}")
    print(f"  • Firefox: {firefox}")
except Exception as e:
    print(f"❌ Error finding dependencies: {e}")

# Test create_xpra_command()
try:
    command = create_xpra_command()
    print(f"\n✅ Generated Xpra command with {len(command)} arguments")
    print(f"Command preview: {' '.join(command[:5])}...")
    
    # Check for problematic arguments
    print(f"\n🔍 Command validation:")
    for i, arg in enumerate(command):
        if 'port' in arg and '{port}' not in arg:
            print(f"  ⚠️  Found hardcoded port in arg {i}: {arg}")
        if arg.startswith('--') and '=' in arg:
            key, value = arg.split('=', 1)
            if not value:
                print(f"  ⚠️  Empty value in arg {i}: {arg}")
    
    # Show key arguments
    key_args = [arg for arg in command if any(x in arg for x in ['bind-tcp', 'html', 'start-child'])]
    print(f"Key arguments: {key_args}")
    
except Exception as e:
    print(f"❌ Error creating command: {e}")
    import traceback
    traceback.print_exc()

=== Testing Firefox Command Creation ===
✅ Found dependencies:
  • Xpra: /usr/bin/xpra
  • Firefox: /usr/bin/firefox

✅ Generated Xpra command with 48 arguments
Command preview: xpra start --bind-tcp=0.0.0.0:{port} --html=on --daemon=no...

🔍 Command validation:
  ⚠️  Empty value in arg 43: --dbus-launch=
Key arguments: ['--bind-tcp=0.0.0.0:{port}', '--html=on', '--start-child=/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher/scripts/firefox-xstartup']


In [11]:
# Test Entry Point System
print("=== Testing Entry Point System ===")

# Check if our entry point is registered
try:
    entry_points = list(pkg_resources.iter_entry_points('jupyter_serverproxy_servers'))
    print(f"Found {len(entry_points)} jupyter-server-proxy entry points:")
    
    firefox_entry = None
    for ep in entry_points:
        print(f"  • {ep.name} = {ep.module_name}:{ep.attrs[0]}")
        if ep.name == 'firefox-desktop':
            firefox_entry = ep
    
    if firefox_entry:
        print(f"\n✅ Found firefox-desktop entry point!")
        
        # Load and test the entry point
        func = firefox_entry.load()
        result = func()
        print(f"Entry point function returns: {type(result)}")
        print(f"Keys: {list(result.keys()) if isinstance(result, dict) else 'Not a dict'}")
    else:
        print("❌ firefox-desktop entry point not found!")
        
except Exception as e:
    print(f"❌ Error testing entry points: {e}")
    import traceback
    traceback.print_exc()

=== Testing Entry Point System ===
Found 1 jupyter-server-proxy entry points:
  • firefox-desktop = jupyterlab_firefox_launcher.server_proxy:launch_firefox

✅ Found firefox-desktop entry point!
Entry point function returns: <class 'dict'>
Keys: ['command', 'timeout', 'new_browser_tab', 'launcher_entry', 'port', 'mappath', 'request_headers_override', 'ready_check']


In [12]:
# Check if jupyter-server-proxy is properly configured
print("=== Checking Jupyter Server Extensions ===")
try:
    from jupyter_server.serverapp import ServerApp
    from jupyter_server.extension.manager import ExtensionManager
    import jupyter_server
    print(f"✅ Jupyter Server version: {jupyter_server.__version__}")
    
    # Check if jupyter-server-proxy is in the extensions
    import sys
    for module_name in sys.modules:
        if 'proxy' in module_name.lower() and 'jupyter' in module_name.lower():
            print(f"Found proxy module: {module_name}")
    
except ImportError as e:
    print(f"❌ Error importing jupyter server: {e}")

# Check if the extension is actually loaded in the current server
try:
    from jupyter_server_proxy import get_server_info
    print("✅ jupyter-server-proxy imported successfully")
    # Try to get server info
    info = get_server_info()
    print(f"Server info keys: {list(info.keys()) if info else 'None'}")
except ImportError as e:
    print(f"❌ Cannot import jupyter-server-proxy: {e}")
except Exception as e:
    print(f"❌ Error getting server info: {e}")

# Check the current notebook server's base URL
try:
    from jupyter_server.base.handlers import JupyterHandler
    from ipykernel.comm import Comm
    import os
    print(f"Current working directory: {os.getcwd()}")
    
    # Get the server root URL if available
    import requests
    import os
    # Try to determine the server URL from environment
    server_url = os.environ.get('JUPYTER_SERVER_ROOT', 'http://localhost:8888')
    print(f"Server URL from environment: {server_url}")
    
except Exception as e:
    print(f"Error checking server info: {e}")

print("\n=== Checking Available Proxies ===")
try:
    # Try to import the proxy configuration loader
    from jupyter_server_proxy import config
    available_proxies = config.get_server_proxy_configuration()
    print(f"Available proxy configurations: {list(available_proxies.keys())}")
    
    if 'firefox-desktop' in available_proxies:
        firefox_config = available_proxies['firefox-desktop']
        print(f"✅ firefox-desktop proxy configuration found!")
        print(f"Config keys: {list(firefox_config.keys())}")
    else:
        print("❌ firefox-desktop proxy configuration NOT found")
        print(f"Available proxies: {list(available_proxies.keys())}")
        
except Exception as e:
    print(f"❌ Error checking proxy configurations: {e}")

=== Checking Jupyter Server Extensions ===
✅ Jupyter Server version: 2.16.0
Found proxy module: jupyterlab_firefox_launcher.server_proxy
❌ Cannot import jupyter-server-proxy: cannot import name 'get_server_info' from 'jupyter_server_proxy' (/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher/.venv/lib/python3.12/site-packages/jupyter_server_proxy/__init__.py)
Current working directory: /home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher
Server URL from environment: http://localhost:8888

=== Checking Available Proxies ===
❌ Error checking proxy configurations: module 'jupyter_server_proxy.config' has no attribute 'get_server_proxy_configuration'


In [13]:
# Check jupyter-server-proxy using the correct API
print("=== Debugging Jupyter Server Proxy API ===")

try:
    import jupyter_server_proxy
    print(f"✅ jupyter-server-proxy version: {jupyter_server_proxy.__version__}")
    
    # Check what's available in the module
    print(f"Available attributes: {[attr for attr in dir(jupyter_server_proxy) if not attr.startswith('_')]}")
    
    # Try the correct way to get server proxy configs
    from jupyter_server_proxy.config import get_server_proxy_specification
    specs = get_server_proxy_specification()
    print(f"Available proxy specs: {list(specs.keys())}")
    
    if 'firefox-desktop' in specs:
        print("✅ firefox-desktop found in proxy specs!")
        firefox_spec = specs['firefox-desktop']
        print(f"Firefox spec type: {type(firefox_spec)}")
        
        # Try to call the spec function
        if callable(firefox_spec):
            try:
                config = firefox_spec()
                print(f"✅ Callable spec returned: {type(config)}")
                print(f"Config keys: {list(config.keys()) if isinstance(config, dict) else 'Not a dict'}")
            except Exception as e:
                print(f"❌ Error calling spec function: {e}")
        else:
            print(f"Spec is not callable: {firefox_spec}")
    else:
        print("❌ firefox-desktop NOT found in proxy specs")
        print(f"Available specs: {list(specs.keys())}")

except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Other error: {e}")

# Let's also check if the server has the extension loaded
print("\n=== Checking Server Extension Loading ===")
try:
    # Check if we can access the current Jupyter server app
    from jupyter_core.paths import jupyter_config_dir
    print(f"Jupyter config dir: {jupyter_config_dir()}")
    
    # Try to see if server extensions are listed
    import subprocess
    result = subprocess.run(['jupyter', 'server', 'extension', 'list'], capture_output=True, text=True)
    print("Server extensions:")
    print(result.stdout)
    if result.stderr:
        print("Stderr:")
        print(result.stderr)
        
except Exception as e:
    print(f"Error checking server extensions: {e}")

print("\n=== Manual Proxy Test ===")
# Let's manually try to create what the proxy should create
try:
    from jupyterlab_firefox_launcher.server_proxy import launch_firefox
    config = launch_firefox()
    print(f"✅ Our launch_firefox() returns: {type(config)}")
    
    # Simulate what jupyter-server-proxy should do
    command = config['command']
    if callable(command):
        test_port = 9998
        actual_command = command(port=test_port)
        print(f"✅ Command with port {test_port}: {' '.join(actual_command[:5])}... ({len(actual_command)} args)")
    else:
        print(f"❌ Command is not callable: {type(command)}")
        
except Exception as e:
    print(f"❌ Error testing our config: {e}")

=== Debugging Jupyter Server Proxy API ===
✅ jupyter-server-proxy version: 4.4.0
Available attributes: ['IconHandler', 'ServerProxyConfig', 'ServersInfoHandler', 'api', 'config', 'get_entrypoint_server_processes', 'handlers', 'load_jupyter_server_extension', 'make_handlers', 'make_server_process', 'rawsocket', 'setup_handlers', 'ujoin', 'unixsock', 'utils', 'websocket']
❌ Import error: cannot import name 'get_server_proxy_specification' from 'jupyter_server_proxy.config' (/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher/.venv/lib/python3.12/site-packages/jupyter_server_proxy/config.py)

=== Checking Server Extension Loading ===
Jupyter config dir: /home/bdx/.jupyter
Server extensions:
Config dir: /home/bdx/.jupyter
Config dir: /home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher/.venv/etc/jupyter
    jupyter_lsp enabled
    - Validating jupyter_lsp...
      jupyter_lsp 2.2.6 OK
    jupyter_server_proxy enabled
    - Validating jupyter_server_proxy...
 

In [14]:
# Test the fixed command function with port parameter
print("=== Testing Fixed Command Function ===")

try:
    from jupyterlab_firefox_launcher.server_proxy import launch_firefox, create_xpra_command
    
    # Test the launch_firefox function
    config = launch_firefox()
    print(f"✅ launch_firefox() returned: {type(config)}")
    print(f"Config keys: {list(config.keys())}")
    
    # Test the command function with a port
    command_func = config['command']
    print(f"✅ Command function: {command_func}")
    
    if callable(command_func):
        test_port = 9999
        try:
            actual_command = command_func(port=test_port)
            print(f"✅ Command with port {test_port} generated successfully!")
            print(f"Command length: {len(actual_command)} arguments")
            print(f"First few args: {' '.join(actual_command[:5])}")
            
            # Check if port is properly substituted
            tcp_bind_arg = None
            for arg in actual_command:
                if arg.startswith('--bind-tcp='):
                    tcp_bind_arg = arg
                    break
            
            if tcp_bind_arg:
                print(f"✅ Port substitution successful: {tcp_bind_arg}")
                if str(test_port) in tcp_bind_arg:
                    print("✅ Port number correctly embedded in command!")
                else:
                    print(f"❌ Port {test_port} not found in {tcp_bind_arg}")
            else:
                print("❌ --bind-tcp argument not found")
                
        except Exception as e:
            print(f"❌ Error calling command function: {e}")
            import traceback
            traceback.print_exc()
    else:
        print(f"❌ Command is not callable: {type(command_func)}")
        
except Exception as e:
    print(f"❌ Error importing or testing: {e}")
    import traceback
    traceback.print_exc()

=== Testing Fixed Command Function ===
✅ launch_firefox() returned: <class 'dict'>
Config keys: ['command', 'timeout', 'new_browser_tab', 'launcher_entry', 'port', 'mappath', 'request_headers_override', 'ready_check']
✅ Command function: <function create_xpra_command at 0x7f36577cbec0>
❌ Error calling command function: create_xpra_command() got an unexpected keyword argument 'port'


Traceback (most recent call last):
  File "/tmp/ipykernel_307385/3165380379.py", line 19, in <module>
    actual_command = command_func(port=test_port)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: create_xpra_command() got an unexpected keyword argument 'port'


In [17]:
# Reload the module to get the fixed version
print("=== Reloading Module to Get Fixed Version ===")

import importlib
import sys

try:
    # Remove the old module from cache
    if 'jupyterlab_firefox_launcher.server_proxy' in sys.modules:
        del sys.modules['jupyterlab_firefox_launcher.server_proxy']
        print("✅ Removed old module from cache")
    
    # Clear the parent package too
    if 'jupyterlab_firefox_launcher' in sys.modules:
        del sys.modules['jupyterlab_firefox_launcher']
        print("✅ Removed parent package from cache")
    
    # Now import fresh
    from jupyterlab_firefox_launcher.server_proxy import launch_firefox, create_xpra_command
    print("✅ Imported fresh modules")
    
    # Test the function signature
    import inspect
    sig = inspect.signature(create_xpra_command)
    print(f"✅ Function signature: create_xpra_command{sig}")
    
    # Test calling it with port
    test_port = 9999
    actual_command = create_xpra_command(test_port)
    print(f"✅ Command generated successfully with {len(actual_command)} arguments")
    
    # Check the bind-tcp argument
    tcp_bind_arg = None
    for arg in actual_command:
        if arg.startswith('--bind-tcp='):
            tcp_bind_arg = arg
            break
    
    if tcp_bind_arg:
        print(f"✅ TCP bind argument: {tcp_bind_arg}")
        if f":{test_port}" in tcp_bind_arg:
            print("✅ Port correctly substituted!")
        else:
            print(f"❌ Port not found in bind argument")
    
    # Test the full config
    config = launch_firefox()
    command_func = config['command']
    test_command = command_func(port=8765)
    print(f"✅ Full config test passed - generated command with {len(test_command)} args")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

=== Reloading Module to Get Fixed Version ===
✅ Removed old module from cache
✅ Removed parent package from cache
✅ Imported fresh modules
✅ Function signature: create_xpra_command(port)
✅ Command generated successfully with 48 arguments
✅ TCP bind argument: --bind-tcp=0.0.0.0:9999
✅ Port correctly substituted!
✅ Full config test passed - generated command with 48 args


In [ ]:
# Final verification - simulate what jupyter-server-proxy will do
print("=== Final End-to-End Test ===")

try:
    from jupyterlab_firefox_launcher.server_proxy import launch_firefox
    
    # Get the configuration that jupyter-server-proxy will use
    config = launch_firefox()
    print("✅ Configuration retrieved")
    
    # Simulate what jupyter-server-proxy does:
    # 1. Get the command function
    command_func = config['command']
    print(f"✅ Command function: {callable(command_func)}")
    
    # 2. Allocate a port (simulate this)
    allocated_port = 12345
    
    # 3. Call the command function with the port
    command_list = command_func(allocated_port)
    print(f"✅ Generated command with {len(command_list)} arguments")
    
    # 4. Check key arguments
    bind_found = False
    html_found = False
    start_child_found = False
    
    for arg in command_list:
        if arg.startswith(f'--bind-tcp=0.0.0.0:{allocated_port}'):
            bind_found = True
            print(f"✅ Bind argument correct: {arg}")
        elif arg == '--html=on':
            html_found = True
            print("✅ HTML5 enabled")
        elif arg.startswith('--start-child='):
            start_child_found = True
            print(f"✅ Start child: {arg}")
    
    if bind_found and html_found and start_child_found:
        print("🎉 ALL TESTS PASSED! The configuration is correct.")
        print("📝 jupyter-server-proxy should now be able to:")
        print("   1. Call launch_firefox() to get config")
        print("   2. Call the command function with an allocated port")
        print("   3. Start the Xpra process with correct arguments")
        print("   4. Serve the HTML5 client at /proxy/firefox-desktop")
    else:
        print("❌ Some critical arguments missing")
        
    # Show first and last few arguments for debugging
    print(f"\n📋 Command preview:")
    print(f"First 5 args: {' '.join(command_list[:5])}")
    print(f"Last 5 args: {' '.join(command_list[-5:])}")
    
except Exception as e:
    print(f"❌ Error in final test: {e}")
    import traceback
    traceback.print_exc()

print("\n=== Next Steps ===")
print("🔄 Restart JupyterLab server to pick up the module changes")
print("🧪 Test /proxy/firefox-desktop endpoint")
print("🚀 Try launching Firefox from the frontend")

=== Final End-to-End Test ===
✅ Configuration retrieved
✅ Command function: True
✅ Generated command with 48 arguments
✅ Bind argument correct: --bind-tcp=0.0.0.0:12345
✅ HTML5 enabled
✅ Start child: --start-child=/home/bdx/allcode/github/vantagecompute/jupyterlab-firefox-launcher/scripts/firefox-xstartup
🎉 ALL TESTS PASSED! The configuration is correct.
📝 jupyter-server-proxy should now be able to:
   1. Call launch_firefox() to get config
   2. Call the command function with an allocated port
   3. Start the Xpra process with correct arguments
   4. Serve the HTML5 client at /proxy/firefox-desktop

📋 Command preview:
First 5 args: xpra start --bind-tcp=0.0.0.0:12345 --html=on --daemon=no
Last 5 args: --dbus-launch= --dbus-proxy=no --remote-logging=no --bandwidth-detection=no --pings=yes

=== Next Steps ===
🔄 Restart JupyterLab server to pick up the module changes
🧪 Test /proxy/firefox-desktop endpoint
🚀 Try launching Firefox from the frontend


: 